In [19]:
import pandas as pd
import numpy as np

print("="*70)
print("TITANIC LABELING VERIFICATION")
print("="*70)

# Load labeled data
labels = pd.read_csv('../data/labeled/titanic_ground_truth.csv')

# Check 1: Row count
print("\n[CHECK 1] Row count verification...")
expected_rows = 89
actual_rows = len(labels)
if actual_rows == expected_rows:
    print(f"✅ PASS: {actual_rows} rows (expected {expected_rows})")
else:
    print(f"❌ FAIL: {actual_rows} rows (expected {expected_rows})")
    exit(1)

# Check 2: Required columns
print("\n[CHECK 2] Required columns verification...")
required_cols = ['PassengerId', 'completeness', 'consistency', 'validity', 'accuracy', 'final_label']
for col in required_cols:
    if col in labels.columns:
        print(f"✅ Column present: {col}")
    else:
        print(f"❌ Column missing: {col}")
        exit(1)

# Check 3: No missing values
print("\n[CHECK 3] Missing values verification...")
for dim in ['completeness', 'consistency', 'validity', 'accuracy', 'final_label']:
    missing_count = labels[dim].isnull().sum()
    if missing_count == 0:
        print(f"✅ {dim}: No missing values")
    else:
        print(f"❌ {dim}: {missing_count} missing values")
        exit(1)

# Check 4: All values are 0 or 1
print("\n[CHECK 4] Value range verification...")
for col in ['completeness', 'consistency', 'validity', 'accuracy', 'final_label']:
    unique_vals = set(labels[col].unique())
    if unique_vals.issubset({0, 1}):
        print(f"✅ {col}: All values are 0 or 1")
    else:
        print(f"❌ {col}: Invalid values found: {unique_vals}")
        exit(1)

# Check 5: Final label logic (≥3 dimensions pass = 1)
print("\n[CHECK 5] Final label logic verification...")
logic_errors = []
for idx, row in labels.iterrows():
    dimensions = [row['completeness'], row['consistency'], row['validity'], row['accuracy']]
    passing_count = sum(dimensions)
    expected_label = 1 if passing_count >= 3 else 0
    if row['final_label'] != expected_label:
        logic_errors.append({
            'row': idx,
            'passing_count': passing_count,
            'actual_label': row['final_label'],
            'expected_label': expected_label
        })

if not logic_errors:
    print(f"✅ All {len(labels)} rows have correct label logic")
else:
    print(f"❌ Found {len(logic_errors)} logic errors:")
    for error in logic_errors:
        print(f"   Row {error['row']}: {error['passing_count']} pass → " +
              f"label={error['actual_label']} (expected {error['expected_label']})")
    exit(1)

# Check 6: Label distribution
print("\n[CHECK 6] Label distribution...")
label_counts = labels['final_label'].value_counts().sort_index()
total = len(labels)
for label_val, count in label_counts.items():
    pct = 100 * count / total
    status = "HIGH QUALITY" if label_val == 1 else "LOW QUALITY"
    print(f"  {status} ({label_val}): {count} rows ({pct:.1f}%)")

print("\n" + "="*70)
print("✅ ALL VERIFICATION CHECKS PASSED")
print("="*70)


TITANIC LABELING VERIFICATION

[CHECK 1] Row count verification...
✅ PASS: 89 rows (expected 89)

[CHECK 2] Required columns verification...
✅ Column present: PassengerId
✅ Column present: completeness
✅ Column present: consistency
✅ Column present: validity
✅ Column present: accuracy
✅ Column present: final_label

[CHECK 3] Missing values verification...
✅ completeness: No missing values
✅ consistency: No missing values
✅ validity: No missing values
✅ accuracy: No missing values
✅ final_label: No missing values

[CHECK 4] Value range verification...
✅ completeness: All values are 0 or 1
✅ consistency: All values are 0 or 1
✅ validity: All values are 0 or 1
✅ accuracy: All values are 0 or 1
✅ final_label: All values are 0 or 1

[CHECK 5] Final label logic verification...
✅ All 89 rows have correct label logic

[CHECK 6] Label distribution...
  LOW QUALITY (0): 8 rows (9.0%)
  HIGH QUALITY (1): 81 rows (91.0%)

✅ ALL VERIFICATION CHECKS PASSED


In [27]:
import pandas as pd

titanic = pd.read_csv('../data/raw/titanic/train.csv')
labels = pd.read_csv('../data/labeled/titanic_ground_truth.csv')
merged = titanic.merge(labels, on='PassengerId', how='inner')

print("=" * 80)
print(f"YOUR 8 LOW QUALITY ROWS")
print("=" * 80)

# Get low quality rows
low_quality = merged[merged['final_label'] == 0]

print(f"\nTotal LOW QUALITY rows: {len(low_quality)}\n")

for idx, (i, row) in enumerate(low_quality.iterrows(), 1):
    dims = [row['completeness'], row['consistency'], row['validity'], row['accuracy']]
    passing = sum(dims)
    
    print(f"\n{idx}. PassengerId={int(row['PassengerId'])}")
    print(f"   Class: {row['Pclass_x'] if 'Pclass_x' in row else 'N/A'}")
    print(f"   Name: {row['Name_x'] if 'Name_x' in row else 'N/A'}")
    print(f"   Age: {row['Age_x'] if 'Age_x' in row else 'N/A'}")
    print(f"   Fare: {row['Fare_x'] if 'Fare_x' in row else 'N/A'}")
    print(f"   ---")
    print(f"   Dimensions: [C={row['completeness']}, Co={row['consistency']}, " +
          f"V={row['validity']}, A={row['accuracy']}]")
    print(f"   Passing: {passing}/4 → final_label = 0 (LOW QUALITY)")
    print(f"   Notes: {row['notes'] if 'notes' in row else 'N/A'}")
    print("-" * 80)


YOUR 8 LOW QUALITY ROWS

Total LOW QUALITY rows: 8


1. PassengerId=65
   Class: 1
   Name: Stewart, Mr. Albert A
   Age: nan
   Fare: 27.7208
   ---
   Dimensions: [C=0, Co=1, V=0, A=1]
   Passing: 2/4 → final_label = 0 (LOW QUALITY)
   Notes: nan
--------------------------------------------------------------------------------

2. PassengerId=470
   Class: 3
   Name: Baclini, Miss. Helene Barbara
   Age: 0.75
   Fare: 19.2583
   ---
   Dimensions: [C=0, Co=0, V=1, A=1]
   Passing: 2/4 → final_label = 0 (LOW QUALITY)
   Notes: nan
--------------------------------------------------------------------------------

3. PassengerId=634
   Class: 1
   Name: Parr, Mr. William Henry Marsh
   Age: nan
   Fare: 0.0
   ---
   Dimensions: [C=0, Co=1, V=0, A=1]
   Passing: 2/4 → final_label = 0 (LOW QUALITY)
   Notes: nan
--------------------------------------------------------------------------------

4. PassengerId=643
   Class: 3
   Name: Skoog, Miss. Margit Elizabeth
   Age: 2.0
   Fare: 27.9
  